# 🪙 获取加密货币行情数据

---

### 🎯 目标
- 用 `ccxt` 连接交易所获取实时价格
- 下载历史 K 线数据 (OHLCV)
- 转成 pandas DataFrame 并画图

### 📌 ccxt 是什么？
- 开源库，统一 API 对接 100+ 个交易所
- 获取行情 **不需要注册、不需要 API Key**
- 换交易所只需改一个单词

In [ ]:
import ccxt
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC']
matplotlib.rcParams['axes.unicode_minus'] = False

print(f"ccxt 版本: {ccxt.__version__}")
print(f"支持 {len(ccxt.exchanges)} 个交易所")
print(f"热门: binance, okx, bybit, gate, coinbase")

## 1️⃣ 连接交易所

In [ ]:
# 如果 Binance 无法访问（国内需要代理），换成 okx 或 bybit
exchange = ccxt.binance({"enableRateLimit": True})
# exchange = ccxt.okx({"enableRateLimit": True})     # 备选1
# exchange = ccxt.bybit({"enableRateLimit": True})    # 备选2

exchange.load_markets()
print(f"✅ 已连接 {exchange.name}")
print(f"📊 可交易品种: {len(exchange.markets)} 个")

## 2️⃣ 获取实时价格

In [ ]:
symbols = ["BTC/USDT", "ETH/USDT", "SOL/USDT"]

print("┌──────────┬────────────┬──────────┬──────────┐")
print("│ 品种     │ 最新价     │ 24h涨跌  │ 24h成交量│")
print("├──────────┼────────────┼──────────┼──────────┤")

for sym in symbols:
    try:
        t = exchange.fetch_ticker(sym)
        emoji = "📈" if (t.get('percentage') or 0) >= 0 else "📉"
        pct = t.get('percentage') or 0
        print(f"│ {sym:<8} │ ${t['last']:>10,.2f}│ {emoji}{pct:>+6.2f}% │ {t.get('baseVolume',0):>8,.0f}│")
    except Exception as e:
        print(f"│ {sym:<8} │ 获取失败   │ 网络问题 │          │")

print("└──────────┴────────────┴──────────┴──────────┘")

## 3️⃣ 下载历史 K 线数据

**OHLCV** = Open(开盘) / High(最高) / Low(最低) / Close(收盘) / Volume(成交量)

In [ ]:
# 获取 BTC 日线，最近 90 天
ohlcv = exchange.fetch_ohlcv("BTC/USDT", timeframe="1d", limit=90)

# 转 DataFrame
df = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "volume"])
df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
df = df.set_index("date").drop("timestamp", axis=1)

print(f"✅ 获取到 {len(df)} 根日线")
print(f"📅 {df.index[0].strftime('%Y-%m-%d')} ~ {df.index[-1].strftime('%Y-%m-%d')}")
df.tail()

## 4️⃣ 画图：BTC 价格走势 + 均线

In [ ]:
df["MA7"] = df["close"].rolling(7).mean()
df["MA30"] = df["close"].rolling(30).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

# 价格 + 均线
ax1.plot(df.index, df["close"], label="BTC 收盘价", color="steelblue", linewidth=1.5)
ax1.plot(df.index, df["MA7"], label="MA7", color="orange", linewidth=1, linestyle="--")
ax1.plot(df.index, df["MA30"], label="MA30", color="red", linewidth=1, linestyle="--")
ax1.fill_between(df.index, df["low"], df["high"], alpha=0.1, color="steelblue")
ax1.set_title("BTC/USDT 日线走势", fontsize=14)
ax1.set_ylabel("价格 ($)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 成交量
colors = ["green" if df["close"].iloc[i] >= df["open"].iloc[i] else "red" for i in range(len(df))]
ax2.bar(df.index, df["volume"], color=colors, alpha=0.7)
ax2.set_ylabel("成交量")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5️⃣ 数据统计

In [ ]:
df["return"] = df["close"].pct_change() * 100

print("📊 BTC 90天统计")
print(f"  最高价: ${df['high'].max():,.2f}")
print(f"  最低价: ${df['low'].min():,.2f}")
print(f"  平均日收益: {df['return'].mean():.4f}%")
print(f"  日波动率: {df['return'].std():.4f}%")
print(f"  最大单日涨: {df['return'].max():.2f}%")
print(f"  最大单日跌: {df['return'].min():.2f}%")

# 收益率分布图
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["return"].dropna(), bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(x=0, color="red", linestyle="--")
ax.set_title("BTC 每日收益率分布", fontsize=14)
ax.set_xlabel("日收益率 (%)")
ax.set_ylabel("天数")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📝 时间周期参考

| timeframe | 含义 | 适用场景 |
|-----------|------|----------|
| `1m` | 1分钟 | 高频/超短线 |
| `5m` | 5分钟 | 短线 |
| `1h` | 1小时 | 波段 |
| `4h` | 4小时 | 中线 |
| `1d` | 1天 | 趋势跟踪 |
| `1w` | 1周 | 长线 |

---
**下一个** → `02_美股数据.ipynb`